In [ ]:
# Installations
!pip install langchain langchain-community langchain-huggingface langchain-chroma langchain-groq sentence-transformers chromadb pymupdf pypdf python-dotenv tqdm

In [ ]:
# Imports
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from tqdm import tqdm

/tmp/ipykernel_4534/159304908.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [ ]:
load_dotenv()
GROQ_API_KEY = os.environ['GROQ_API_KEY']
if GROQ_API_KEY is None:
  raise ValueError("GROQ_API_KEY is not set")

In [ ]:
# Load Embedding Model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},  # Change to "cuda" if you have an NVIDIA GPU
    encode_kwargs={"normalize_embeddings": True}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Initializing LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [ ]:
# Data Path of PDFs used
DATA_PATH = "/content/drive/MyDrive/RAG_Article/data"

In [ ]:
# Load the documents
documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyMuPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages.")

Loaded 36 pages.


In [ ]:
# Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks.")

Created 30 chunks.


In [ ]:
# ChromaDB Path
CHROMA_DB_DIR = "/content/drive/MyDrive/RAG_Article/chroma_db"

In [ ]:
# Initialize Vector Database
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=CHROMA_DB_DIR
)

print("Vector database created successfully!")

Vector database created successfully!


In [ ]:
# Add documents to the vector database
collection = vector_store._collection

print(f"Total chunks indexed: {collection.count()}")

Total chunks indexed: 60


In [ ]:
# Relevant chunks retrieval object
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [ ]:
# System Prompt
prompt = ChatPromptTemplate.from_template("""
You are an intelligent AI assistant.

Answer ONLY using the provided context.

If the answer is not present in the context, simply say:
"I couldn't find that information in the provided documents."

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
# Formatting of retrieved documents that the LLM can clearly understand
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# User Prompt
query = "List the different opeartions on arrays"

In [ ]:
# Retrieval of relevant chunks based on user prompt
retrieved_docs = retriever.invoke(query)

In [ ]:
# System Prompt + User Prompt
context = format_docs(retrieved_docs)

formatted_prompt = prompt.format(
    context=context,
    question=query
)

In [ ]:
print(formatted_prompt)

Human: 
You are an intelligent AI assistant.

Answer ONLY using the provided context.

If the answer is not present in the context, simply say:
"I couldn't find that information in the provided documents."

Context:
Operations on Arrays
• Searching a element
• Insertion of new element
• Deletion of required element
• Modification of an element
• Merging of arrays

Operations on Arrays
• Searching a element
• Insertion of new element
• Deletion of required element
• Modification of an element
• Merging of arrays

Fig shows array representation of stack:

Fig shows array representation of stack:

Question:
List the different opeartions on arrays

Answer:



In [ ]:
# LLM Call & Response
response = llm.invoke(formatted_prompt)
print(response.content)

- Searching an element  
- Insertion of a new element  
- Deletion of a required element  
- Modification of an element  
- Merging of arrays
